# Datavidia 11.0 — Task-1 · Notebook Submission

Notebook ini menghasilkan `submission_final.csv` **tanpa melatih ulang model apa pun**.
Seluruh parameter sudah tersimpan di folder `models/`.

| Masukan | Keluaran |
|---|---|
| `data/raw/` (8 tabel train + 8 tabel test) | `submissions/submission_final.csv` |
| `models/` (5 LightGBM + efek acak Stage-2 + bobot ensemble) | 362.245 baris, 2 kolom |

## Langkah

1. Bangun fitur test dari data mentah — `src/build_features.py` (tanpa label, tanpa pelatihan).
2. Muat 5 model `lgb_f*.txt`, prediksi logits, rata-ratakan.
3. Terapkan koreksi efek acak entitas dari `stage2_*_corr.npy`.
4. Gabungkan dengan bobot `ensemble_weight.npy`.
5. `argmax` → label → CSV, lalu validasi terhadap `sample_submission.csv`.

**Waktu jalan: ± 50 menit pada CPU 2 core** — prediksi 5 model ± 9 menit per model (20.000 baris diuji dalam 7,3 detik), feature engineering ± 5 menit bila berkas fitur belum ada. Pada mesin dengan core lebih banyak, prediksi jauh lebih cepat.

> Tidak ada pemanggilan `lgb.train`, tidak ada pembacaan `status_penanganan` dari data test,
> dan tidak ada penyesuaian parameter berdasarkan data test.


## 0. Persiapan

In [1]:
import json, os, sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

T0 = time.time()
def log(m): print(f"[{time.time()-T0:7.1f}s] {m}", flush=True)

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

MODELS, OUT = ROOT / "models", ROOT / "submissions"
OUT.mkdir(exist_ok=True)
print("ROOT:", ROOT)

ROOT: C:\Datavidia\terpakai


In [2]:
import lightgbm as lgb, numpy as np, pandas as pd
from scipy.special import softmax

spec = json.load(open(MODELS / "feature_spec.json"))
FEATS, CATS, CLASSES = spec["feature_cols"], spec["cat_cols"], spec["classes"]
print(f"{len(FEATS)} fitur | {len(CATS)} kategorikal | {len(CLASSES)} kelas")
print("kelas:", CLASSES)

190 fitur | 10 kategorikal | 7 kelas
kelas: ['gagal_antar_awal', 'klaim_hilang', 'konfirmasi_alamat', 'normal', 'retur_pengirim', 'rusak_perjalanan', 'tertunda_operasional']


## 1. Fitur test dari data mentah

Fungsi `build_features.main()` membangun `feat_train.pkl` dan `feat_test.pkl` dari `data/raw`.
Profil entitas dihitung pada gabungan train+test **tanpa memakai label** (leave-one-out untuk
courier dan customer), sehingga tidak ada kebocoran target. Bila berkas fitur sudah ada,
langkah ini dilewati.

In [3]:
p_tr, p_te = ROOT / "data/processed/feat_train.pkl", ROOT / "data/processed/feat_test.pkl"
if not (p_tr.exists() and p_te.exists()):
    log("membangun fitur dari data mentah…")
    import build_features
    build_features.main()
else:
    log("berkas fitur sudah ada — dilewati")

te = pd.read_pickle(p_te)
log(f"test features: {te.shape}")
assert len(te) == 362245

[   25.5s] berkas fitur sudah ada — dilewati


[   27.5s] test features: (362245, 195)


## 2. Muat model (tanpa pelatihan)

In [4]:
boosters = []
for f in range(5):
    b = lgb.Booster(model_file=str(MODELS / f"lgb_f{f}.txt"))
    assert b.feature_name() == FEATS, f"urutan fitur model {f} tidak cocok"
    boosters.append(b)
log(f"{len(boosters)} model dimuat — {boosters[0].num_trees()} pohon per model, "
    f"{len(boosters[0].feature_name())} fitur")

[   36.4s] 5 model dimuat — 2989 pohon per model, 190 fitur


## 3. Prediksi Stage-1 (rata-rata 5 fold)

In [5]:
X = te[FEATS].copy()
for c in CATS:
    X[c] = pd.Categorical(X[c].astype(str), categories=spec["categories"][c])
for c in FEATS:
    if c not in CATS:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32").fillna(spec["fill_value"])
log(f"matriks prediksi: {X.shape}")

CACHE = ROOT / "data/processed/stage1_test_logits_rebuilt.npy"
if CACHE.exists() and np.load(CACHE).shape == (len(X), len(CLASSES)):
    stage1 = np.load(CACHE)
    log("menggunakan cache prediksi yang sudah dihitung")
else:
    stage1 = np.zeros((len(X), len(CLASSES)), dtype=np.float32)
    for i, b in enumerate(boosters):
        stage1 += b.predict(X, raw_score=True).astype(np.float32) / len(boosters)
        np.save(CACHE, stage1)          # cadangan bila proses terputus
        log(f"model {i} selesai")

ref = np.load(MODELS / "stage1_test_logits.npy")
log(f"max|Δ| vs logits Stage-1 tersimpan: {np.abs(stage1 - ref).max():.3e}")

[   44.4s] matriks prediksi: (362245, 190)


[  752.0s] model 0 selesai


[ 1469.4s] model 1 selesai


[ 2164.4s] model 2 selesai


[ 2858.6s] model 3 selesai


[ 3489.8s] model 4 selesai


[ 3490.0s] max|Δ| vs logits Stage-1 tersimpan: 0.000e+00


## 4. Koreksi Stage-2 (efek acak entitas)

`u` sudah ditaksir saat pelatihan dan disimpan per entitas per kelas. Di sini hanya dilakukan
pencarian nilai: entitas tak dikenal → koreksi 0.

In [6]:
G = {
    "hub": te["hub_id"].astype(str).values,
    "courier": te["courier_id"].astype(str).values,
    "customer": te["customer_id"].astype(str).values,
}
corr = {n: np.load(MODELS / f"stage2_{n}_corr.npy", allow_pickle=True).item() for n in G}

stage2 = stage1.copy()
for n, g in G.items():
    for c in range(len(CLASSES)):
        m = corr[n][c]
        stage2[:, c] += np.array([m.get(x, 0.0) for x in g], dtype=np.float32)
    log(f"koreksi {n} diterapkan")

log(f"max|Δ| vs logits Stage-2 tersimpan: {np.abs(stage2 - np.load(MODELS / 'stage2_test_logits.npy')).max():.3e}")

[ 3496.3s] koreksi hub diterapkan


[ 3498.5s] koreksi courier diterapkan


[ 3502.9s] koreksi customer diterapkan


[ 3503.0s] max|Δ| vs logits Stage-2 tersimpan: 4.768e-07


## 5. Ensemble & argmax

In [7]:
w = float(np.load(MODELS / "ensemble_weight.npy")[0])
space = spec.get("ensemble_space", "prob")
log(f"bobot Stage-1 w={w:.2f} (ruang {space})")

if space == "prob":
    p = w * softmax(stage1, 1) + (1 - w) * softmax(stage2, 1)
else:
    p = w * stage1 + (1 - w) * stage2

pred = p.argmax(1)
log(f"prediksi: {len(pred):,} baris, keyakinan rata-rata {p.max(1).mean():.4f}")

[ 3503.0s] bobot Stage-1 w=0.00 (ruang prob)


[ 3503.3s] prediksi: 362,245 baris, keyakinan rata-rata 0.7019


## 6. Tulis & validasi submission

In [8]:
sub = pd.DataFrame({
    "shipment_id": te["shipment_id"].values,
    "status_penanganan": [CLASSES[i] for i in pred],
})
path = OUT / "submission_final.csv"
sub.to_csv(path, index=False)
log(f"tersimpan: {path}")

sample = pd.read_csv(ROOT / "data/raw/sample_submission.csv")
assert list(sub.columns) == list(sample.columns), "kolom tidak cocok"
assert len(sub) == len(sample), "jumlah baris tidak cocok"
assert set(sub.shipment_id) == set(sample.shipment_id), "himpunan shipment_id tidak cocok"
assert sub.status_penanganan.isin(CLASSES).all(), "ada label di luar kelas"
print("\nVALID: kolom, jumlah baris, himpunan shipment_id, dan label — semua sesuai sample_submission")
print(sub.head().to_string(index=False))
print("\ndistribusi prediksi:")
print(sub.status_penanganan.value_counts(normalize=True).round(4).to_string())

[ 3504.6s] tersimpan: C:\Datavidia\terpakai\submissions\submission_final.csv



VALID: kolom, jumlah baris, himpunan shipment_id, dan label — semua sesuai sample_submission
 shipment_id    status_penanganan
SHP-5A64RENJ               normal
SHP-BUZSJTV9    konfirmasi_alamat
SHP-EGUZBJHQ               normal
SHP-YWBRSQLZ    konfirmasi_alamat
SHP-JS89CKME tertunda_operasional

distribusi prediksi:
status_penanganan
normal                  0.4168
konfirmasi_alamat       0.2350
tertunda_operasional    0.1611
gagal_antar_awal        0.0787
retur_pengirim          0.0483
rusak_perjalanan        0.0393
klaim_hilang            0.0207


## 7. Bukti reproduksibilitas

Dibandingkan dengan submission acuan yang dihasilkan pipeline yang sama saat pelatihan.

In [9]:
ref_sub = OUT / "reference_submission.csv"
if ref_sub.exists():
    old = pd.read_csv(ref_sub)
    same = (old.sort_values("shipment_id").reset_index(drop=True)["status_penanganan"].values ==
            sub.sort_values("shipment_id").reset_index(drop=True)["status_penanganan"].values)
    print(f"baris identik dengan acuan: {same.mean():.6%}  ({same.sum():,}/{len(same):,})")
else:
    print("acuan tidak disertakan")

baris identik dengan acuan: 100.000000%  (362,245/362,245)
